In [1]:
# Run this once only. After installation, you can comment this cell out.
!pip install yfinance pandas numpy matplotlib plotly duckdb

Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 10.0 MB/s  0:00:00
   ---------------------------------------- 0.0/13.1 MB ? eta -:--:--
   ------ --------------------------------- 2.1/13.1 MB 11.7 MB/s eta 0:00:01
   -------------- ------------------------- 4.7/13.1 MB 11.3 MB/s eta 0:00:01
   --------------------- ------------------ 7.1/13.1 MB 11.4 MB/s eta 0:00:01
   ---------------------------- ----------- 9.4/13.1 MB 11.5 MB/s eta 0:00:01
   ----------------------------------- ---- 11.5/13.1 MB 11.0 MB/s 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\wizar\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import duckdb
import os
from datetime import datetime

print("All imports successful")
print(f"yfinance version: {yf.__version__}")
print(f"pandas version: {pd.__version__}")

All imports successful
yfinance version: 1.2.0
pandas version: 2.3.2


In [3]:
SYMBOLS = {
    "BTC": "BTC-USD",
    "ETH": "ETH-USD",
}

# yfinance hard limits — don't change these
INTERVAL_LIMITS = {
    "1h": "729d",   # stays within the 730-day window safely
    "1d": "5y",     # full 5-year history
}

END_DATE = datetime.today().strftime("%Y-%m-%d")

DATA_DIR = "../data/raw/"
DB_PATH  = "../data/tradingbot.duckdb"

os.makedirs(DATA_DIR, exist_ok=True)

print(f"Fetching up to: {END_DATE}")
print(f"1h window : last 729 days")
print(f"1d window : last 5 years")

Fetching up to: 2026-03-18
1h window : last 729 days
1d window : last 5 years


In [4]:
def fetch_ohlcv(ticker: str, period: str, interval: str) -> pd.DataFrame:
    """
    Fetches OHLCV using 'period' instead of a fixed start date.
    This respects yfinance's interval-specific history limits.
    period examples: '729d', '5y', '60d'
    """
    print(f"Fetching {ticker} | interval={interval} | period={period}")

    df = yf.download(
        tickers     = ticker,
        period      = period,      # ← key change: period not start date
        interval    = interval,
        auto_adjust = True,
        progress    = False
    )

    if df.empty:
        raise ValueError(f"No data returned for {ticker} at {interval}")

    # Flatten multi-level columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = [c.lower() for c in df.columns]
    df.index.name = "timestamp"
    df.dropna(subset=["open", "high", "low", "close", "volume"], inplace=True)

    print(f"  → {len(df)} candles | {df.index[0]} to {df.index[-1]}")
    return df


# Fetch 1h — last 729 days
raw_data_1h = {}
for name, ticker in SYMBOLS.items():
    raw_data_1h[name] = fetch_ohlcv(ticker, period="729d", interval="1h")

# Fetch daily — last 5 years (for macro trend context)
raw_data_1d = {}
for name, ticker in SYMBOLS.items():
    raw_data_1d[name] = fetch_ohlcv(ticker, period="5y", interval="1d")

print("\nAll data fetched successfully.")

Fetching BTC-USD | interval=1h | period=729d
  → 17453 candles | 2024-03-20 00:00:00+00:00 to 2026-03-18 12:00:00+00:00
Fetching ETH-USD | interval=1h | period=729d
  → 17450 candles | 2024-03-20 00:00:00+00:00 to 2026-03-18 12:00:00+00:00
Fetching BTC-USD | interval=1d | period=5y
  → 1827 candles | 2021-03-18 00:00:00 to 2026-03-18 00:00:00
Fetching ETH-USD | interval=1d | period=5y
  → 1827 candles | 2021-03-18 00:00:00 to 2026-03-18 00:00:00

All data fetched successfully.


In [5]:
def resample_to_4h(df_1h: pd.DataFrame) -> pd.DataFrame:
    """
    yfinance doesn't serve 4h candles directly.
    We build them by resampling 1h candles — perfectly accurate.
    """
    df_4h = df_1h.resample("4h").agg({
        "open"  : "first",   # first candle's open
        "high"  : "max",     # highest point in the 4h window
        "low"   : "min",     # lowest point in the 4h window
        "close" : "last",    # last candle's close
        "volume": "sum"      # total volume traded
    }).dropna()
    
    return df_4h


raw_data_4h = {}

for name, df in raw_data_1h.items():
    raw_data_4h[name] = resample_to_4h(df)
    print(f"{name} 4h: {len(raw_data_4h[name])} candles")

BTC 4h: 4366 candles
ETH 4h: 4366 candles


In [6]:
def sanity_check(df: pd.DataFrame, name: str, interval: str):
    """
    Checks your data for common problems before you use it.
    Catches issues now rather than during model training.
    """
    print(f"\n{'='*40}")
    print(f"  {name} | {interval}")
    print(f"{'='*40}")
    print(f"  Shape        : {df.shape}")
    print(f"  Date range   : {df.index[0]} → {df.index[-1]}")
    print(f"  Missing vals : {df.isnull().sum().sum()}")
    print(f"  Duplicate idx: {df.index.duplicated().sum()}")
    
    # Check for zero or negative prices (bad data)
    bad_prices = (df[["open","high","low","close"]] <= 0).any().any()
    print(f"  Bad prices   : {bad_prices}")
    
    # Check OHLC logic: high >= low, high >= open/close
    ohlc_valid = (
        (df["high"] >= df["low"]).all() and
        (df["high"] >= df["open"]).all() and
        (df["high"] >= df["close"]).all()
    )
    print(f"  OHLC valid   : {ohlc_valid}")
    print(f"  Avg volume   : {df['volume'].mean():,.0f}")


for name in SYMBOLS:
    sanity_check(raw_data_1h[name], name, "1h")
    sanity_check(raw_data_4h[name], name, "4h")


  BTC | 1h
  Shape        : (17453, 5)
  Date range   : 2024-03-20 00:00:00+00:00 → 2026-03-18 12:00:00+00:00
  Missing vals : 0
  Duplicate idx: 0
  Bad prices   : False
  OHLC valid   : True
  Avg volume   : 527,485,096

  BTC | 4h
  Shape        : (4366, 5)
  Date range   : 2024-03-20 00:00:00+00:00 → 2026-03-18 12:00:00+00:00
  Missing vals : 0
  Duplicate idx: 0
  Bad prices   : False
  OHLC valid   : True
  Avg volume   : 2,108,611,402

  ETH | 1h
  Shape        : (17450, 5)
  Date range   : 2024-03-20 00:00:00+00:00 → 2026-03-18 12:00:00+00:00
  Missing vals : 0
  Duplicate idx: 0
  Bad prices   : False
  OHLC valid   : True
  Avg volume   : 330,347,947

  ETH | 4h
  Shape        : (4366, 5)
  Date range   : 2024-03-20 00:00:00+00:00 → 2026-03-18 12:00:00+00:00
  Missing vals : 0
  Duplicate idx: 0
  Bad prices   : False
  OHLC valid   : True
  Avg volume   : 1,320,332,496


In [7]:
import os

# Save folder — sits inside your ALGO2 project
DATA_DIR = "./data/raw"
os.makedirs(DATA_DIR, exist_ok=True)

def save_csv(df: pd.DataFrame, filename: str):
    path = os.path.join(DATA_DIR, filename)
    df.to_csv(path)
    print(f"Saved: {path}  ({len(df)} rows)")

# Save all six tables as CSVs
save_csv(raw_data_1h["BTC"], "btc_1h.csv")
save_csv(raw_data_1h["ETH"], "eth_1h.csv")
save_csv(raw_data_4h["BTC"], "btc_4h.csv")
save_csv(raw_data_4h["ETH"], "eth_4h.csv")
save_csv(raw_data_1d["BTC"], "btc_1d.csv")
save_csv(raw_data_1d["ETH"], "eth_1d.csv")

print("\nAll data saved as CSVs.")

Saved: ./data/raw\btc_1h.csv  (17453 rows)
Saved: ./data/raw\eth_1h.csv  (17450 rows)
Saved: ./data/raw\btc_4h.csv  (4366 rows)
Saved: ./data/raw\eth_4h.csv  (4366 rows)
Saved: ./data/raw\btc_1d.csv  (1827 rows)
Saved: ./data/raw\eth_1d.csv  (1827 rows)

All data saved as CSVs.


In [8]:
import os

# See exactly where Python thinks it is right now
print(f"Current working directory: {os.getcwd()}")

# Try creating a test file right here
test_path = os.path.join(os.getcwd(), "test_write.txt")
try:
    with open(test_path, "w") as f:
        f.write("test")
    print(f"Write works at: {test_path}")
    os.remove(test_path)
except Exception as e:
    print(f"Write FAILED: {e}")

# Try creating the data folder
data_path = os.path.join(os.getcwd(), "data", "raw")
try:
    os.makedirs(data_path, exist_ok=True)
    print(f"Folder created: {data_path}")
    print(f"Folder exists : {os.path.exists(data_path)}")
except Exception as e:
    print(f"Folder creation FAILED: {e}")

Current working directory: s:\ML models\algo2
Write works at: s:\ML models\algo2\test_write.txt
Folder created: s:\ML models\algo2\data\raw
Folder exists : True


In [11]:
# Read back from DB to confirm everything saved correctly
con = duckdb.connect(DB_PATH)

for table in ["btc_1h", "eth_1h", "btc_4h", "eth_4h"]:
    result = con.execute(f"""
        SELECT 
            COUNT(*)        AS rows,
            MIN(timestamp)  AS earliest,
            MAX(timestamp)  AS latest
        FROM {table}
    """).df()
    print(f"\n{table}:")
    print(result.to_string(index=False))

con.close()

CatalogException: Catalog Error: Table with name btc_1h does not exist!
Did you mean "sqlite_schema"?

LINE 6:         FROM btc_1h
                     ^

In [12]:
def plot_candlestick(df: pd.DataFrame, title: str):
    """
    Interactive candlestick chart with volume.
    Hover over any candle to see OHLCV values.
    """
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.75, 0.25],
        vertical_spacing=0.03
    )

    # Candlestick chart
    fig.add_trace(go.Candlestick(
        x     = df.index,
        open  = df["open"],
        high  = df["high"],
        low   = df["low"],
        close = df["close"],
        name  = "Price"
    ), row=1, col=1)

    # Volume bars — green if close > open, red otherwise
    colors = ["#1D9E75" if c >= o else "#D85A30"
              for c, o in zip(df["close"], df["open"])]

    fig.add_trace(go.Bar(
        x      = df.index,
        y      = df["volume"],
        name   = "Volume",
        marker_color = colors,
        opacity = 0.7
    ), row=2, col=1)

    fig.update_layout(
        title           = title,
        xaxis_rangeslider_visible = False,
        height          = 600,
        template        = "plotly_dark",
        showlegend      = False
    )

    fig.show()


# Plot last 90 days of 1h data for both symbols
for name in SYMBOLS:
    df_plot = raw_data_1h[name].last("90D")
    plot_candlestick(df_plot, f"{name}/USD — 1h — Last 90 Days")

C:\Users\wizar\AppData\Local\Temp\ipykernel_22104\2570058863.py:48: FutureWarning: last is deprecated and will be removed in a future version. Please create a mask and filter using `.loc` instead
  df_plot = raw_data_1h[name].last("90D")


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [13]:
def print_stats(df: pd.DataFrame, name: str, interval: str):
    returns = df["close"].pct_change().dropna()
    
    print(f"\n{name} | {interval}")
    print(f"  Total candles     : {len(df):,}")
    print(f"  Avg daily return  : {returns.mean()*100:.4f}%")
    print(f"  Volatility (std)  : {returns.std()*100:.4f}%")
    print(f"  Max single candle : +{returns.max()*100:.2f}%")
    print(f"  Min single candle : {returns.min()*100:.2f}%")
    print(f"  All-time high     : ${df['high'].max():,.2f}")
    print(f"  All-time low      : ${df['low'].min():,.2f}")

for name in SYMBOLS:
    print_stats(raw_data_1h[name], name, "1h")


BTC | 1h
  Total candles     : 17,453
  Avg daily return  : 0.0021%
  Volatility (std)  : 0.5135%
  Max single candle : +5.08%
  Min single candle : -4.92%
  All-time high     : $126,183.23
  All-time low      : $49,578.89

ETH | 1h
  Total candles     : 17,450
  Avg daily return  : 0.0007%
  Volatility (std)  : 0.7373%
  Max single candle : +10.13%
  Min single candle : -11.90%
  All-time high     : $4,953.02
  All-time low      : $1,387.61
